# Validation (phần 2) — Bảng mạng lưới: Supply, Transfer, Last-mile B2B, DOS, Lead time

Tiếp nối `Validation.ipynb`. Đọc file gốc trong `Raw Data/`, kiểm tra, làm sạch và xuất sang `Model - Data used/`.
Phạm vi: Indonesia, B2B, 53 kho trong `facility_master.parquet`, 3.347 nhóm khách trong `shipto_group_map.parquet`.
Mọi chi phí quy về **giá 2025** (Decision_Log 5.7): supply / transfer chia 1,1546; last-mile B2B chia 1,328.

## 0. Setup

In [1]:
# ===== SETUP =====
import pandas as pd                       # xử lý bảng
import numpy as np                        # tính toán
from pathlib import Path                  # ghép đường dẫn
from collections import deque             # hàng đợi cho kiểm tra "đi tới được" (BFS)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

RAW = Path("../Raw Data")                 # data gốc (chỉ đọc)
BL = RAW / "0,1_Baseline2025"             # mạng hiện tại 2025
GF = RAW / "3_GF_Unconstrained"           # kịch bản 2030, nhiều tuyến hơn
OUT = Path("../Model - Data used")        # nơi lưu data đã clean

DEFLATE_FM = 1.1546                       # hệ số trượt giá first/mid mile 2025 -> 2030 (supply, transfer)
DEFLATE_LM = 1.3280                       # hệ số trượt giá last-mile B2B 2025 -> 2030

# Phạm vi model đã chốt ở phần 1
fac = pd.read_parquet(OUT / "facility_master.parquet")                    # 53 kho
F = set(fac["LocationID"])                                                # tập mã kho
G = set(pd.read_parquet(OUT / "shipto_group_map.parquet")["ShipToGroupID"])  # tập mã nhóm khách
print(f"Kho: {len(F)} | Nhóm khách: {len(G):,}")

Kho: 53 | Nhóm khách: 3,347


## 2.7 Supply (nhà máy → NDC)

In [2]:
# ===== 2.7 SUPPLY =====
# Dùng bản BL: chỉ nhà máy hiện tại -> NDC Jatake. Bản GF có thêm nhà máy / NDC Batang (đã loại, Decision_Log 4.1a)
sup_bl = pd.read_parquet(BL / "ENO_CostSupply.parquet")
sup_gf = pd.read_parquet(GF / "ENO_CostSupply.parquet")
print("BL:\n", sup_bl, "\n\nGF:\n", sup_gf)

supply = sup_bl[sup_bl["DestinationID"].isin(F)].copy()                  # chỉ tuyến tới NDC trong 53 kho
assert (supply["CostPerPallet"] == 0).all()                               # nhà máy cạnh NDC -> chi phí 0
supply.to_parquet(OUT / "cost_supply.parquet", index=False)
print(f"\nĐã lưu cost_supply.parquet | {len(supply)} tuyến, tất cả chi phí = 0")

BL:
   OriginID DestinationID  CostPerPallet
0       J1           NDC            0.0
1       J2           NDC            0.0
2       J4           NDC            0.0
3       J6           NDC            0.0 

GF:
   OriginID DestinationID  CostPerPallet
0       J1           NDC            0.0
1       J2           NDC            0.0
2       J4           NDC            0.0
3       J6           NDC            0.0
4       B1          NDC3            0.0
5       B2          NDC3            0.0
6       B4          NDC3            0.0
7       B6          NDC3            0.0

Đã lưu cost_supply.parquet | 4 tuyến, tất cả chi phí = 0


## 2.8 Transfer (kho → kho)

In [3]:
# ===== 2.8a TRANSFER: kiểm tra =====
tr_gf = pd.read_parquet(GF / "ENO_CostTransfer.parquet")                  # mọi tuyến kho -> kho (giá 2030)
tr_bl = pd.read_parquet(BL / "ENO_CostTransfer.parquet")                  # chỉ tuyến đang chạy (giá 2025)
tr = tr_gf[tr_gf["OriginID"].isin(F) & tr_gf["DestinationID"].isin(F)].copy()   # chỉ tuyến giữa 53 kho
print(f"Tuyến GF: {len(tr_gf):,} -> trong 53 kho: {len(tr):,} | đang chạy (IsExistingFlow): {tr['IsExistingFlow'].sum()}")
print("Tuyến tự đi tới chính nó:", (tr["OriginID"] == tr["DestinationID"]).sum())
print("Tuyến trùng:", tr.duplicated(["OriginID", "DestinationID"]).sum(), "| CostPerPallet <= 0:", (tr["CostPerPallet"] <= 0).sum())

# GF có đúng là BL × 1,1546 không (trên các tuyến chung)
chung = tr.merge(tr_bl, on=["OriginID", "DestinationID"], suffixes=("", "_BL"))
chung["ty_le"] = chung["CostPerPallet"] / chung["CostPerPallet_BL"]
print(f"\nTuyến chung với BL: {len(chung)} | tỷ lệ GF/BL trung vị = {chung['ty_le'].median():.4f}")
print("Tuyến lệch khỏi 1,1546:")
print(chung.loc[(chung["ty_le"] - DEFLATE_FM).abs() > 0.001, ["OriginID", "DestinationID", "CostPerPallet", "CostPerPallet_BL", "ty_le"]])

# Loại kho đi -> loại kho đến
loai = fac.set_index("LocationID")["FacilityType"]
print("\nLoại kho đi × loại kho đến:")
print(pd.crosstab(tr["OriginID"].map(loai), tr["DestinationID"].map(loai)))

# Mọi kho có nhận được hàng từ NDC không (đi theo các tuyến transfer)
ke = {}
for o, d in zip(tr["OriginID"], tr["DestinationID"]):
    ke.setdefault(o, []).append(d)
da_toi, hang_doi = {"NDC"}, deque(["NDC"])
while hang_doi:
    n = hang_doi.popleft()
    for m in ke.get(n, []):
        if m not in da_toi:
            da_toi.add(m); hang_doi.append(m)
print(f"\nKho nhận được hàng từ NDC: {len(da_toi)}/{len(F)} | không tới được: {sorted(F - da_toi)}")

Tuyến GF: 3,000 -> trong 53 kho: 369 | đang chạy (IsExistingFlow): 73
Tuyến tự đi tới chính nó: 0
Tuyến trùng: 0 | CostPerPallet <= 0: 0

Tuyến chung với BL: 73 | tỷ lệ GF/BL trung vị = 1.1546
Tuyến lệch khỏi 1,1546:
   OriginID DestinationID  CostPerPallet  CostPerPallet_BL     ty_le
58      NDC       MWH_D14   6.275830e+05      6.565927e+05  0.955818
60      NDC       MWH_D19   1.392365e+06      1.325768e+06  1.050233

Loại kho đi × loại kho đến:
DestinationID  DC Direct  DC Satellite  DEPO  RDC
OriginID                                         
DC Direct             20            34    17   24
DC Satellite          24            67    11   38
NDC                    7            18     2    9
RDC                   24            47     9   18

Kho nhận được hàng từ NDC: 53/53 | không tới được: []


In [4]:
# ===== 2.8b TRANSFER: clean + xuất =====
tr["CostPerPallet_2030"] = tr["CostPerPallet"]                            # giữ giá gốc để đối chiếu
tr["CostPerPallet"] = tr["CostPerPallet_2030"] / DEFLATE_FM               # giá 2025 (model dùng cột này)
assert not tr.duplicated(["OriginID", "DestinationID"]).any()
assert (tr["CostPerPallet"] > 0).all() and tr["DistanceInKM"].notna().all()
tr.to_parquet(OUT / "cost_transfer.parquet", index=False)
print(f"Đã lưu cost_transfer.parquet | {len(tr)} tuyến | giá 2025 trung vị {tr['CostPerPallet'].median():,.0f} IDR/pallet")

Đã lưu cost_transfer.parquet | 369 tuyến | giá 2025 trung vị 2,009,593 IDR/pallet


In [ ]:
# ===== 2.8c TRANSFER: 369 tuyến gồm những tuyến nào =====
# Mục đích: hiểu tập tuyến công ty tạo ra (Assumption 4.7: giữ tuyến đang chạy + tuyến mới cùng đảo / cặp đảo được phép)
assert "MainIsland" in fac.columns                                        # cần cột đảo trong facility_master
dao = fac.set_index("LocationID")["MainIsland"]                           # mã kho -> đảo

t = tr.copy()
t["LoaiDi"] = t["OriginID"].map(loai)                                     # loại kho đi (NDC, RDC, DC Direct, ...)
t["LoaiDen"] = t["DestinationID"].map(loai)                               # loại kho đến
t["DaoDi"] = t["OriginID"].map(dao)                                       # đảo kho đi
t["DaoDen"] = t["DestinationID"].map(dao)                                 # đảo kho đến
t["CungDao"] = t["DaoDi"] == t["DaoDen"]                                  # True = tuyến nội đảo
t["NguonGoc"] = np.where(t["IsExistingFlow"], "Đang chạy 2025", "Tuyến mới (GF)")   # tuyến có sẵn hay công ty sinh thêm

# (1) Tuyến đang chạy vs tuyến mới
print("(1) Đang chạy / mới:")
print(t["NguonGoc"].value_counts(), "\n")

# (2) Đảo đi × đảo đến (đường chéo = nội đảo)
print("(2) Đảo đi × đảo đến:")
print(pd.crosstab(t["DaoDi"], t["DaoDen"], margins=True), "\n")

# (3) Các tuyến liên đảo: cặp đảo nào, bao nhiêu tuyến, mấy tuyến đang chạy
lien_dao = t[~t["CungDao"]]
print(f"(3) Tuyến liên đảo: {len(lien_dao)} / {len(t)}")
print(lien_dao.groupby(["DaoDi", "DaoDen"])
      .agg(so_tuyen=("OriginID", "size"), dang_chay=("IsExistingFlow", "sum"))
      .sort_values("so_tuyen", ascending=False), "\n")

# (4) Loại kho đi -> đến, tách đang chạy / mới
print("(4) Loại kho đi -> đến, theo nguồn gốc tuyến:")
print(pd.crosstab([t["LoaiDi"], t["LoaiDen"]], t["NguonGoc"], margins=True), "\n")

# (5) Tập tuyến có "đầy đủ" trong nội đảo không: so với số cặp kho cùng đảo có thể có
so_kho_dao = dao.value_counts()                                           # số kho mỗi đảo
cap_co_the = (so_kho_dao * (so_kho_dao - 1)).rename("cap_co_the")         # số cặp có hướng cùng đảo (n × (n-1))
cap_co = t[t["CungDao"]].groupby("DaoDi").size().rename("cap_trong_bang")  # số tuyến nội đảo thực có trong bảng
print("(5) Mức phủ tuyến nội đảo:")
print(pd.concat([so_kho_dao.rename("so_kho"), cap_co_the, cap_co], axis=1).fillna(0).astype(int), "\n")

# (6) Khoảng cách theo nguồn gốc tuyến (kiểm tra có ngưỡng km ngầm định không, vd DC -> DEPO <= 200 km)
print("(6) Khoảng cách (km) theo loại kho đến và nguồn gốc:")
print(t.groupby(["LoaiDen", "NguonGoc"])["DistanceInKM"].describe()[["count", "min", "50%", "max"]].round(0))

In [ ]:
# ===== 2.8d TRANSFER: nếu bỏ tuyến cùng cấp / ngược tầng thì có đủ không =====
# Cấp kho (theo nhãn 2025): NDC = 1; RDC, DC Direct = 2 (cùng nhận thẳng từ NDC); DC Satellite = 3; DEPO = 4
CAP = {"NDC": 1, "RDC": 2, "DC Direct": 2, "DC Satellite": 3, "DEPO": 4}
t["CapDi"] = t["LoaiDi"].map(CAP)                                         # cấp kho đi
t["CapDen"] = t["LoaiDen"].map(CAP)                                       # cấp kho đến
assert t[["CapDi", "CapDen"]].notna().all().all()                         # mọi loại kho đều có cấp
t["Huong"] = np.select([t["CapDi"] < t["CapDen"], t["CapDi"] == t["CapDen"]],
                       ["Đi xuống", "Cùng cấp"], default="Ngược lên")     # hướng của tuyến
print("Số tuyến theo hướng (đang chạy 2025 / mới):")
print(pd.crosstab(t["Huong"], t["NguonGoc"], margins=True), "\n")

def toi_duoc(bang):
    """Trả về tập kho nhận được hàng từ NDC khi chỉ dùng các tuyến trong `bang` (BFS)."""
    ke = bang.groupby("OriginID")["DestinationID"].apply(list).to_dict()  # kho đi -> danh sách kho đến
    da_toi, q = {"NDC"}, deque(["NDC"])
    while q:
        for m in ke.get(q.popleft(), []):
            if m not in da_toi:
                da_toi.add(m); q.append(m)
    return da_toi

# Tuyến vào rẻ nhất của mỗi kho ở bản đầy đủ (để so chi phí)
re_nhat_day_du = t.groupby("DestinationID")["CostPerPallet"].min()

PHUONG_AN = {
    "Đầy đủ": t,
    "Bỏ cùng cấp": t[t["Huong"] != "Cùng cấp"],
    "Chỉ đi xuống": t[t["Huong"] == "Đi xuống"],
}
for ten, b in PHUONG_AN.items():
    toi = toi_duoc(b)                                                     # kho nhận được hàng
    mat_cu = int(t["IsExistingFlow"].sum() - b["IsExistingFlow"].sum())   # số tuyến đang chạy 2025 bị bỏ
    re_nhat = b.groupby("DestinationID")["CostPerPallet"].min()           # tuyến vào rẻ nhất sau khi bỏ
    so = pd.concat([re_nhat_day_du.rename("truoc"), re_nhat.rename("sau")], axis=1).dropna()
    bi_anh_huong = so[so["sau"] > so["truoc"] * 1.001]                    # kho mà tuyến vào rẻ nhất bị đắt lên
    print(f"--- {ten}: {len(b)} tuyến | kho nhận được hàng {len(toi)}/{len(F)} | "
          f"tuyến đang chạy 2025 bị bỏ: {mat_cu}/{int(t['IsExistingFlow'].sum())} | "
          f"kho có tuyến vào rẻ nhất đắt lên: {len(bi_anh_huong)}")
    if len(toi) < len(F):
        print("   Không tới được:", sorted(F - toi))
    if len(bi_anh_huong):
        x = bi_anh_huong.assign(LoaiKho=bi_anh_huong.index.map(loai), Dao=bi_anh_huong.index.map(dao),
                                tang_pct=(bi_anh_huong["sau"] / bi_anh_huong["truoc"] - 1) * 100)
        print(x.sort_values("tang_pct", ascending=False).round(0).to_string(), "\n")

## 2.9 Last-mile B2B (kho → nhóm khách)

In [5]:
# ===== 2.9 LAST-MILE B2B: gộp 2 bảng + clean + xuất =====
# Decision_Log 5.7: gộp bảng GF và bảng root (giá giống hệt trên tuyến chung), chỉ giữ tuyến 53 kho -> 3.347 nhóm, chia 1,328
lm_gf = pd.read_parquet(GF / "ENO_CostLastMileB2B.parquet")
lm_unc = pd.read_parquet(RAW / "ENO_CostLastMileB2B_Unconstaint.parquet")
lm = pd.concat([lm_gf, lm_unc]).drop_duplicates(["OriginID", "DestinationID"])
lm = lm[lm["OriginID"].isin(F) & lm["DestinationID"].isin(G)].copy()

lm["CostPerPallet_2030"] = lm["CostPerPallet"]
lm["CostPerPallet"] = lm["CostPerPallet_2030"] / DEFLATE_LM               # giá 2025

so_kho = lm.groupby("DestinationID")["OriginID"].nunique()                 # mỗi nhóm có mấy kho giao tới
print(f"Tuyến: {len(lm):,} | nhóm được phục vụ: {lm['DestinationID'].nunique():,}/{len(G):,} | "
      f"kho có tuyến: {lm['OriginID'].nunique()}/{len(F)}")
print("Số kho giao tới mỗi nhóm (số kho: số nhóm):", so_kho.value_counts().sort_index().to_dict())

assert set(lm["DestinationID"]) == G                                      # mọi nhóm đều có ít nhất 1 kho
assert (lm["CostPerPallet"] > 0).all() and not lm.duplicated(["OriginID", "DestinationID"]).any()
lm.to_parquet(OUT / "cost_lastmile_b2b.parquet", index=False)
print(f"Đã lưu cost_lastmile_b2b.parquet | giá 2025 trung vị {lm['CostPerPallet'].median():,.0f} IDR/pallet")

Tuyến: 14,770 | nhóm được phục vụ: 3,347/3,347 | kho có tuyến: 53/53
Số kho giao tới mỗi nhóm (số kho: số nhóm): {1: 217, 2: 548, 3: 531, 4: 370, 5: 463, 6: 619, 7: 459, 8: 119, 9: 20, 10: 1}
Đã lưu cost_lastmile_b2b.parquet | giá 2025 trung vị 2,934,845 IDR/pallet


## 2.11 DOS (số ngày tồn kho)

In [6]:
# ===== 2.11 DOS: gộp về 1 giá trị / kho (trung vị theo nhóm sản phẩm) =====
dos = pd.read_parquet(GF / "ENO_DOS.parquet")                             # DOS theo kho × nhóm sản phẩm
d = dos[dos["LocationID"].isin(F)]
print(f"Dòng DOS của 53 kho: {len(d):,} | kho có DOS: {d['LocationID'].nunique()}/{len(F)} | DOS <= 0: {(d['DOS'] <= 0).sum():,}")

dos_kho = d.groupby("LocationID")["DOS"].median().rename("DOS").reset_index()    # trung vị (Decision_Log 4.5)
dos_kho["FacilityType"] = dos_kho["LocationID"].map(fac.set_index("LocationID")["FacilityType"])
print("\nDOS (ngày) theo loại kho:")
print(dos_kho.groupby("FacilityType")["DOS"].describe()[["count", "min", "50%", "max"]])

assert len(dos_kho) == len(F)                                             # đủ 53 kho
assert ((dos_kho["DOS"] > 0) | (dos_kho["FacilityType"] == "DEPO")).all()  # chỉ DEPO được có DOS = 0 (cross-dock)
dos_kho.to_parquet(OUT / "dos.parquet", index=False)
print("Đã lưu dos.parquet")

Dòng DOS của 53 kho: 7,526 | kho có DOS: 53/53 | DOS <= 0: 2,556

DOS (ngày) theo loại kho:
              count  min   50%   max
FacilityType                        
DC Direct       7.0  7.0   9.8  28.0
DC Satellite   18.0  7.0   9.8  29.4
DEPO           18.0  0.0   0.0   0.0
NDC             1.0  7.0   7.0   7.0
RDC             9.0  8.4  11.2  23.8
Đã lưu dos.parquet


## 2.12 Lead time + khoảng cách

In [7]:
# ===== 2.12a LEAD TIME: kiểm tra =====
lt = pd.read_parquet(RAW / "LeadTime_AllFlow.parquet")
print(f"Dòng: {len(lt):,} | trùng hoàn toàn: {lt.duplicated().sum():,} | trùng (Origin, Dest): {lt.duplicated(['OriginID', 'DestinationID']).sum():,}")
lt = lt.drop_duplicates()                                                 # các dòng trùng giống hệt nhau -> bỏ
assert not lt.duplicated(["OriginID", "DestinationID"]).any()

# Tuyến model cần: transfer (kho -> kho) và last-mile (kho -> nhóm)
can = pd.concat([tr[["OriginID", "DestinationID"]].assign(Loai="Transfer"),
                 lm[["OriginID", "DestinationID"]].assign(Loai="LastMile")])
can = can.merge(lt, on=["OriginID", "DestinationID"], how="left")
print("\nTuyến model cần có lead time không:")
print(can.groupby("Loai")["TotalLeadTimeInDays"].agg(so_tuyen="size", thieu=lambda s: s.isna().sum()))
print("\nLead time (ngày) và khoảng cách (km) theo loại tuyến:")
print(can.groupby("Loai")[["TotalLeadTimeInDays", "DistanceInKM"]].describe().T.round(2))
print("\nTuyến có chặng biển:", (can["SeaLeadTimeInDays"] > 0).groupby(can["Loai"]).sum().to_dict())

Dòng: 293,161 | trùng hoàn toàn: 154,217 | trùng (Origin, Dest): 154,217



Tuyến model cần có lead time không:
          so_tuyen  thieu
Loai                     
LastMile     14770      0
Transfer       369      0

Lead time (ngày) và khoảng cách (km) theo loại tuyến:
Loai                       LastMile  Transfer
TotalLeadTimeInDays count  14770.00    369.00
                    mean       0.24      0.87
                    std        0.66      1.51
                    min        0.00      0.03
                    25%        0.09      0.25
                    50%        0.15      0.53
                    75%        0.25      0.91
                    max       15.75     11.82
DistanceInKM        count  14770.00    369.00
                    mean     176.94    670.83
                    std      196.96    506.59
                    min        0.70     27.10
                    25%       86.30    257.74
                    50%      140.32    563.21
                    75%      192.61    964.65
                    max     5270.10   2569.64

Tuyến có chặng biển: 

In [8]:
# ===== 2.12b LEAD TIME: clean + xuất (chỉ tuyến model dùng) =====
COT = ["OriginID", "DestinationID", "Loai", "FlowType", "DistanceType", "DistanceInKM",
       "InlandDistanceInKM", "SeaDistanceKM", "InlandLeadTimeInDays", "SeaLeadTimeInDays",
       "TotalLeadTimeInDays", "UrbanArea"]
lead = can[COT].copy()
lead["SeaDistanceKM"] = lead["SeaDistanceKM"].fillna(0)                   # tuyến không có biển -> 0 km biển
assert lead["TotalLeadTimeInDays"].notna().all() and lead["DistanceInKM"].notna().all()
lead.to_parquet(OUT / "lead_time.parquet", index=False)
print(f"Đã lưu lead_time.parquet | {len(lead):,} tuyến (transfer + last-mile)")

Đã lưu lead_time.parquet | 15,139 tuyến (transfer + last-mile)


In [ ]:
# ===== 2.12c LEAD TIME: bảng lead time có khoảng cách cho các cặp KHÔNG có trong bảng cost không =====
# Mục đích: nếu có, ta có thể tự tính cost cho tuyến còn thiếu bằng công thức của công ty (đơn giá pallet-km × km)
cap_lt = set(zip(lt["OriginID"], lt["DestinationID"]))                    # mọi cặp (đi, đến) có trong bảng lead time

# (1) Kho -> kho: mọi cặp có hướng giữa 53 kho (không tính tự đi tới chính nó)
cap_kho = {(o, d) for o in F for d in F if o != d}                        # 53 × 52 = 2.756 cặp
cap_tr = set(zip(tr["OriginID"], tr["DestinationID"]))                    # 369 cặp có cost
print(f"(1) Kho -> kho: {len(cap_kho):,} cặp có thể | có cost: {len(cap_tr)} | "
      f"có trong bảng lead time: {len(cap_kho & cap_lt):,} | "
      f"thiếu cost NHƯNG có lead time: {len((cap_kho - cap_tr) & cap_lt):,}")

# (1b) Tách theo đảo: các cặp thiếu cost nhưng có lead time nằm ở đâu
thieu = pd.DataFrame(sorted((cap_kho - cap_tr) & cap_lt), columns=["OriginID", "DestinationID"])
if len(thieu):
    thieu["DaoDi"] = thieu["OriginID"].map(dao)
    thieu["DaoDen"] = thieu["DestinationID"].map(dao)
    print("   Cặp thiếu cost nhưng có lead time, theo đảo đi × đảo đến:")
    print(pd.crosstab(thieu["DaoDi"], thieu["DaoDen"], margins=True))

# (2) Kho -> nhóm khách: mọi cặp 53 kho × 3.347 nhóm
so_cap_lm = len(F) * len(G)                                               # 177.391 cặp
lt_lm = lt[lt["OriginID"].isin(F) & lt["DestinationID"].isin(G)]          # dòng lead time thuộc kho -> nhóm
print(f"\n(2) Kho -> nhóm: {so_cap_lm:,} cặp có thể | có cost: {len(lm):,} | "
      f"có trong bảng lead time: {len(lt_lm):,}")

# (3) Bảng lead time còn chứa những loại điểm nào khác (để biết nó được sinh theo quy tắc gì)
loai_diem = lambda s: np.where(s.isin(F), "Kho (53)", np.where(s.isin(G), "Nhóm khách", "Khác"))
print("\n(3) Loại điểm đi × điểm đến trong bảng lead time:")
print(pd.crosstab(loai_diem(lt["OriginID"]), loai_diem(lt["DestinationID"]), margins=True))

## Tổng kết file đã xuất

In [9]:
# ===== DANH SÁCH FILE TRONG Model - Data used =====
for f in sorted(OUT.glob("*.parquet")):
    df = pd.read_parquet(f)
    print(f"{f.name:28s} {len(df):>12,} dòng × {df.shape[1]} cột")

cost_lastmile_b2b.parquet          14,770 dòng × 4 cột
cost_supply.parquet                     4 dòng × 3 cột
cost_transfer.parquet                 369 dòng × 6 cột
customer_master.parquet            46,461 dòng × 16 cột
dos.parquet                            53 dòng × 3 cột
facility_master.parquet                53 dòng × 21 cột
lead_time.parquet                  15,139 dòng × 12 cột
product_master.parquet              4,660 dòng × 31 cột
shipto_group_map.parquet           34,430 dòng × 2 cột


so_b2b.parquet                 24,084,062 dòng × 10 cột
